
# TEI English XML Extractor (with `<pb/>` preserved)

This notebook extracts the **English sections** from a TEI XML file and writes a new TEI that preserves all markup, including `<pb/>` page breaks.  
It uses **lxml** and deep-copies the detected English blocks (`xml:lang="en"` or `<div type="translation">`).

**Features**
- Keeps the original characters (no text normalization).
- Preserves **all descendants** (including `<pb/>`).  
- Optional `drop_notes=True` to remove `<note>` elements only.
- Optional `copy_header=True` to include the source `<teiHeader>`.


In [11]:

# If needed, install lxml (usually already available)
# You can comment this out if lxml is installed.
# %pip install -q lxml
print("Environment ready.")


Environment ready.


In [16]:

from lxml import etree as LET
from copy import deepcopy
from pathlib import Path

TEI_NS = "http://www.tei-c.org/ns/1.0"
XML_NS = "http://www.w3.org/XML/1998/namespace"
NS = {"tei": TEI_NS, "xml": XML_NS}

def is_en(val: str) -> bool:
    if not val:
        return False
    v = val.strip().lower()
    return v == "en" or v.startswith("en-")

def find_top_english(root):
    """Return top-most English nodes in document order."""
    candidates = []
    # any element with xml:lang or @lang = en*
    for el in root.xpath(".//*[@xml:lang] | .//*[@lang]"):
        lang = el.get(f"{{{XML_NS}}}lang") or el.get("lang")
        if is_en(lang):
            candidates.append(el)
    # any explicit translation div
    candidates += root.xpath(".//tei:div[@type='translation']", namespaces=NS)

    # unique preserve order
    seen = set()
    uniq = []
    for el in candidates:
        if el in seen:
            continue
        seen.add(el)
        uniq.append(el)

    # keep top-most only
    cand_set = set(uniq)
    tops = []
    for el in uniq:
        parent = el.getparent()
        keep = True
        while parent is not None:
            if parent in cand_set:
                keep = False
                break
            parent = parent.getparent()
        if keep:
            tops.append(el)

    # fallback if nothing detected
    if not tops:
        b = root.find(".//tei:body", namespaces=NS)
        if b is not None:
            tops = [b]
        else:
            t = root.find(".//tei:text", namespaces=NS)
            if t is not None:
                tops = [t]
    return tops

def build_english_tei(root, copy_header=True, drop_notes=False):
    """Build a new TEI with only the English blocks in <text><body>.

    - copy_header: include source <teiHeader> if present

    - drop_notes: remove <note> descendants only (keeps <pb/>)

    """
    tei = LET.Element(f"{{{TEI_NS}}}TEI", nsmap={None: TEI_NS, "xml": XML_NS})

    if copy_header:
        header = root.find(".//tei:teiHeader", namespaces=NS)
        if header is not None:
            tei.append(deepcopy(header))

    text = LET.SubElement(tei, f"{{{TEI_NS}}}text")
    body = LET.SubElement(text, f"{{{TEI_NS}}}body")

    english_blocks = find_top_english(root)
    if not english_blocks:
        raise RuntimeError("No English blocks detected. Ensure xml:lang='en' or <div type='translation'> is present.")

    for blk in english_blocks:
        cp = deepcopy(blk)
        if drop_notes:
            for n in list(cp.xpath(".//tei:note", namespaces=NS)):
                p = n.getparent()
                if p is not None:
                    p.remove(n)
        body.append(cp)
    return tei
print("Helpers loaded.")    


Helpers loaded.


In [17]:

# === Configure these paths ===
input_xml = Path("../GRC_misc/phi0474.phil049.king1.xml")   # change to your file
output_xml = input_xml.with_name(input_xml.stem + "_english2.xml")

print("Input:", input_xml)
print("Output:", output_xml)


Input: ../GRC_misc/phi0474.phil049.king1.xml
Output: ../GRC_misc/phi0474.phil049.king1_english.xml


In [14]:

# Parse source TEI and build English-only TEI
parser = LET.XMLParser(remove_blank_text=False, recover=False, huge_tree=True)
tree = LET.parse(str(input_xml), parser)
root = tree.getroot()

out_tei = build_english_tei(root, copy_header=True, drop_notes=False)

# Write file
with open(output_xml, "wb") as f:
    f.write(LET.tostring(out_tei, xml_declaration=True, encoding="UTF-8", pretty_print=True))

print("Wrote:", output_xml)


Wrote: ../GRC_misc/phi0474.phil049.king1_english.xml


In [10]:

# Verify that <pb/> count is preserved inside the English scope
src_root = LET.parse(str(input_xml), parser).getroot()
out_root = LET.parse(str(output_xml), parser).getroot()

# Find the English blocks in source (top-most)
english_blocks = find_top_english(src_root)

def count_pb_in_blocks(blocks):
    total = 0
    for el in blocks:
        total += len(el.xpath(".//tei:pb", namespaces=NS))
    return total

src_pb = count_pb_in_blocks(english_blocks)
out_pb = len(out_root.xpath(".//tei:pb", namespaces=NS))

print("Source English <pb/>:", src_pb)
print("Output <pb/>:", out_pb)
assert src_pb == out_pb, "Mismatch in <pb/> counts!"
print("OK: <pb/> preserved one-to-one.")


Source English <pb/>: 218
Output <pb/>: 218
OK: <pb/> preserved one-to-one.


In [23]:

# OPTIONAL: rebuild without <note> (still keeps <pb/>)
drop_notes_output = input_xml.with_name(input_xml.stem + "_english_nonotes.xml")

out_nonotes = build_english_tei(root, copy_header=True, drop_notes=True)
with open(drop_notes_output, "wb") as f:
    f.write(LET.tostring(out_nonotes, xml_declaration=True, encoding="UTF-8", pretty_print=True))

print("Wrote (no notes):", drop_notes_output)


Wrote (no notes): ../GRC_misc/phi0474.phil049.king1_english_nonotes.xml
